# 01 · List every settled market on both platforms
Kalshi and Polymarket both publish market data openly (no keys). This notebook downloads the list of settled markets for
the categories both platforms cover in the same yes/no form: **NBA, NFL, MLB and NHL games** and **Fed rate decisions**.
Runs in GitHub Actions. Prices come later, in notebook 03, only for matched pairs.

### Settings

In [1]:
from pathlib import Path
import json, time
import pandas as pd
import requests

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)
KALSHI = "https://api.elections.kalshi.com/trade-api/v2"
GAMMA = "https://gamma-api.polymarket.com"
HEADERS = {"User-Agent": "oddsgap research project (github.com/AdityaKhetarpal/oddsgap)", "Accept": "application/json"}
WINDOW = ("2025-01-01", "2026-10-31")                      # Kalshi close dates in the locked window
KALSHI_SERIES = {"KXNBAGAME": "NBA", "KXNFLGAME": "NFL", "KXMLBGAME": "MLB", "KXNHLGAME": "NHL", "KXFEDDECISION": "Fed"}
POLY_SERIES = {"NBA": 10345, "NFL": 12185, "MLB": 3, "NHL": 10346}   # Polymarket's league series ids

### A polite downloader
Waits a little between requests, retries when the server says "too many requests" (HTTP 429), and gives up after 5 tries.

In [2]:
def get(url, params=None):
    for attempt in range(5):
        r = requests.get(url, params=params, headers=HEADERS, timeout=60)
        if r.status_code == 429 or r.status_code >= 500:
            time.sleep(2 ** attempt)                          # back off: 1, 2, 4, 8 seconds
            continue
        time.sleep(0.15)
        if r.status_code == 404:
            return None
        r.raise_for_status()
        return r.json()
    raise RuntimeError(f"gave up on {url}")

### Kalshi: settled markets
Kalshi keeps recent settled markets at `/markets` and moves older ones to `/historical/markets`. I page through both
for each series and keep one row per market.

In [3]:
def kalshi_markets(series):
    rows = []
    for path, extra in [("/markets", {"status": "settled"}), ("/historical/markets", {})]:
        cursor = None
        while True:
            page = get(KALSHI + path, {"series_ticker": series, "limit": 1000, **extra, **({"cursor": cursor} if cursor else {})})
            if not page:
                break
            rows += page.get("markets", [])
            cursor = page.get("cursor")
            if not cursor or not page.get("markets"):
                break
    return rows

rows = []
for series, league in KALSHI_SERIES.items():
    got = kalshi_markets(series)
    print(series, len(got), "markets")
    for m in got:
        m["league"] = league
        m["series_ticker"] = series
    rows += got
data = pd.DataFrame(rows).drop_duplicates("ticker")
print(len(data), "Kalshi markets")
data[["ticker", "event_ticker", "yes_sub_title", "close_time", "result", "volume_fp"]].head()

KXNBAGAME 3452 markets


KXNFLGAME 1096 markets


KXMLBGAME 9582 markets


KXNHLGAME 3880 markets


KXFEDDECISION 155 markets
17936 Kalshi markets


,ticker,event_ticker,yes_sub_title,close_time,result,volume_fp
0,KXNBAGAME-26NOV17PORMIN-POR,KXNBAGAME-26NOV17PORMIN,Portland,2026-11-18T03:37:40Z,no,5177558.16
1,KXNBAGAME-26NOV17PORMIN-MIN,KXNBAGAME-26NOV17PORMIN,Minnesota,2026-11-18T03:37:40Z,yes,5553565.69
2,KXNBAGAME-26NOV17NYKTOR-TOR,KXNBAGAME-26NOV17NYKTOR,Toronto,2026-11-18T02:57:57Z,no,5127735.61
3,KXNBAGAME-26NOV17NYKTOR-NYK,KXNBAGAME-26NOV17NYKTOR,New York,2026-11-18T02:57:57Z,yes,8171037.43
4,KXNBAGAME-26NOV17UTALAC-UTA,KXNBAGAME-26NOV17UTALAC,Utah,2026-11-18T05:50:02Z,yes,5113647.73


### Keep the columns I need, inside the window

In [4]:
keep = ["ticker", "event_ticker", "series_ticker", "league", "title", "yes_sub_title", "open_time", "close_time",
        "result", "volume_fp", "volume", "status"]
data = data[[c for c in keep if c in data.columns]].copy()
data["close_time"] = pd.to_datetime(data["close_time"], utc=True, format="ISO8601")
data["open_time"] = pd.to_datetime(data["open_time"], utc=True, format="ISO8601")
data["volume"] = pd.to_numeric(data.get("volume_fp", data.get("volume")), errors="coerce")   # contracts traded ($1 each at most)
data = data[(data["close_time"] >= pd.Timestamp(WINDOW[0], tz="UTC")) & (data["close_time"] < pd.Timestamp(WINDOW[1], tz="UTC") + pd.Timedelta(days=1))]
data = data[data["result"].isin(["yes", "no"])]                                              # drop voided markets
data.drop(columns=["volume_fp"], errors="ignore").to_parquet(RAW / "kalshi_markets.parquet", index=False)
data.groupby("league").size()

league
Fed      75
MLB    9368
NBA    3190
NFL     966
NHL    3636
dtype: int64

### Polymarket: closed game events per league
Polymarket groups each league's games into a series. Each game event has a slug like `nba-phx-lal-2025-10-03` and a
"moneyline" market with one outcome per team.

In [5]:
def poly_events(series_id):
    out, cursor = [], None
    while True:
        page = get(GAMMA + "/events/keyset", {"series_id": series_id, "closed": "true", "limit": 200,
                                              **({"after_cursor": cursor} if cursor else {})})
        if not page:
            break
        out += page.get("events", [])
        cursor = page.get("next_cursor")
        if not cursor or not page.get("events"):
            break
    return out

rows = []
for league, sid in POLY_SERIES.items():
    evs = poly_events(sid)
    print(league, len(evs), "events")
    for e in evs:
        for m in e.get("markets", []):
            if m.get("sportsMarketType") != "moneyline":
                continue                                          # spreads and totals have no Kalshi twin here
            rows.append({"league": league, "event_slug": e.get("slug"), "event_title": e.get("title"),
                         "game_start": m.get("gameStartTime") or e.get("startTime"), "question": m.get("question"),
                         "outcomes": m.get("outcomes"), "outcome_prices": m.get("outcomePrices"),
                         "token_ids": m.get("clobTokenIds"), "volume": m.get("volumeNum"),
                         "closed_time": m.get("closedTime"), "end_date": m.get("endDate"), "condition_id": m.get("conditionId")})
data = pd.DataFrame(rows)
print(len(data), "Polymarket moneyline markets")
data.head()

NBA 1890 events


NFL 593 events


MLB 11500 events


NHL 1822 events
8815 Polymarket moneyline markets


,league,event_slug,event_title,game_start,question,outcomes,outcome_prices,token_ids,volume,closed_time,end_date,condition_id
0,NBA,nba-phi-nyk-2025-10-02,76ers vs. Knicks,2025-10-02 16:00:00+00,76ers vs. Knicks,"[""76ers"", ""Knicks""]","[""0"", ""1""]","[""10775890349961162154638110618292068659867810...",6423.016398,2025-10-03 06:32:20+00,2025-10-09T16:00:00Z,0x97ddcbda2ac98ec09f9b8c374814e8ccf4e4f444ae45...
1,NBA,nba-phx-lal-2025-10-03,Suns vs. Lakers,2025-10-04 02:00:00+00,Suns vs. Lakers,"[""Suns"", ""Lakers""]","[""1"", ""0""]","[""11514618982429731370716451366828800228857052...",313285.705076,2025-10-04 06:38:01+00,2025-10-11T02:00:00Z,0xe4a44179a29ec6727007b54038892d6a3ac0993635ff...
2,NBA,nba-nyk-phi-2025-10-04,Knicks vs. 76ers,2025-10-04 15:00:00+00,Knicks vs. 76ers,"[""Knicks"", ""76ers""]","[""1"", ""0""]","[""10111973957792308840100972931210359509184764...",92255.751683,2025-10-04 20:20:58+00,2025-10-11T15:00:00Z,0x2f7c5d2589500c9b4c1f3a1c9d8dbc79224b870c4f84...
3,NBA,nba-orl-mia-2025-10-04,Magic vs. Heat,2025-10-05 00:00:00+00,Magic vs. Heat,"[""Magic"", ""Heat""]","[""1"", ""0""]","[""66490370726406005483173897021619926846060816...",43565.692967,2025-10-05 05:57:32+00,2025-10-12T00:00:00Z,0x332ccf6a70732b039b3bc872c59d622050838086c3b2...
4,NBA,nba-min-den-2025-10-04,Timberwolves vs. Nuggets,2025-10-05 01:00:00+00,Timberwolves vs. Nuggets,"[""Timberwolves"", ""Nuggets""]","[""1"", ""0""]","[""35782468627171912975679670944209383931413195...",54077.398567,2025-10-05 05:58:38+00,2025-10-12T01:00:00Z,0xac889c3dacf0c9f347deda8a631a86a2f983b5b031ac...


### Polymarket: Fed decision events
Polymarket has one event per Fed meeting ("Fed decision in October?") with one yes/no market per outcome.

In [6]:
fed = []
for q in ["Fed decision in", "Fed Decision in"]:
    page = get(GAMMA + "/public-search", {"q": q, "events_status": "closed", "limit_per_type": 100})
    for e in (page or {}).get("events", []):
        if not e.get("title", "").lower().startswith("fed decision in"):
            continue
        for m in e.get("markets", []):
            fed.append({"league": "Fed", "event_slug": e.get("slug"), "event_title": e.get("title"), "game_start": None,
                        "question": m.get("question"), "group_title": m.get("groupItemTitle"), "outcomes": m.get("outcomes"),
                        "outcome_prices": m.get("outcomePrices"), "token_ids": m.get("clobTokenIds"), "volume": m.get("volumeNum"),
                        "closed_time": m.get("closedTime"), "end_date": m.get("endDate") or e.get("endDate"),
                        "condition_id": m.get("conditionId")})
fed = pd.DataFrame(fed).drop_duplicates("condition_id")
print(len(fed), "Polymarket Fed outcome markets in", fed["event_slug"].nunique(), "meetings")
data = pd.concat([data, fed], ignore_index=True)
data.to_parquet(RAW / "poly_markets.parquet", index=False)
data.groupby("league").size()

96 Polymarket Fed outcome markets in 21 meetings


league
Fed      96
MLB    5108
NBA    1685
NFL     200
NHL    1822
dtype: int64

### Polymarket team codes
The slug uses Polymarket's team codes. I save each league's team list to check codes against Kalshi's.

In [7]:
teams = []
for league in ["nba", "nfl", "mlb", "nhl"]:
    for t in get(GAMMA + "/teams", {"league": league, "limit": 500}) or []:
        teams.append({"league": league.upper(), "name": t.get("name"), "abbreviation": (t.get("abbreviation") or "").lower()})
pd.DataFrame(teams).to_parquet(RAW / "poly_teams.parquet", index=False)
print(len(teams), "teams")

137 teams


### Write down where the data came from

In [8]:
from datetime import date
(ROOT / "data" / "SOURCES.md").write_text(f"""# Data sources (downloaded {date.today()})

- **Kalshi trade API v2** (public market data, no key): `{KALSHI}` — `/markets`, `/historical/markets`, `/events`,
  `/markets/candlesticks`, `/historical/markets/{{ticker}}/candlesticks`.
- **Polymarket Gamma API** (public): `{GAMMA}` — `/events/keyset` by league series, `/public-search`, `/teams`.
- **Polymarket Data API** (public): `https://data-api.polymarket.com/v2/prices-history` (3-hour buckets, kept since 2022).
- Window: Kalshi markets that closed {WINDOW[0]} to {WINDOW[1]}. Descriptive research only; no trading.
""")
print("saved data/SOURCES.md")

saved data/SOURCES.md
